In [2]:
import pandas as pd
data_path="../data/processed/main_questions_cleaned.csv"
clean_df=pd.read_csv(data_path)

print("Dataset shape:", clean_df.shape)
print("Unique students:", clean_df["user_id"].nunique())
print("Unique order IDs:", clean_df["order_id"].nunique())

Dataset shape: (275458, 7)
Unique students: 4217
Unique order IDs: 275458


In [3]:
from sklearn.model_selection import train_test_split

student_ids=clean_df["user_id"].unique()

train_student_ids,test_student_ids=train_test_split(
    student_ids,
    test_size=0.20,
    random_state=42,
)

print("Training students:", len(train_student_ids))
print("Test students:", len(test_student_ids))

Training students: 3373
Test students: 844


In [4]:
train_df = clean_df[
    clean_df["user_id"].isin(train_student_ids)
].copy()

test_df = clean_df[
    clean_df["user_id"].isin(test_student_ids)
].copy()

print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)

Training shape: (219983, 7)
Test shape: (55475, 7)


In [5]:
train_users = set(train_df["user_id"].unique())
test_users = set(test_df["user_id"].unique())

print("Training students:", len(train_users))
print("Test students:", len(test_users))
print("Overlapping students:", len(train_users & test_users))
print("Total interactions:", len(train_df) + len(test_df))

Training students: 3373
Test students: 844
Overlapping students: 0
Total interactions: 275458


In [6]:
from sklearn.model_selection import KFold

kf=KFold(n_splits=5,shuffle=True,random_state=42)
folds=[]

for fold_number,(train_idx,val_idx) in enumerate(
    kf.split(train_student_ids),start=1
):
  fold_train_ids=train_student_ids[train_idx]
  fold_val_ids=train_student_ids[val_idx]

  folds.append((fold_train_ids,fold_val_ids))
  print(f"Fold{fold_number}: "
        f"Training students={len(fold_train_ids)} "
         f"Validation students = {len(fold_val_ids)}"
        )



Fold1: Training students=2698 Validation students = 675
Fold2: Training students=2698 Validation students = 675
Fold3: Training students=2698 Validation students = 675
Fold4: Training students=2699 Validation students = 674
Fold5: Training students=2699 Validation students = 674


In [7]:
fold_datasets=[]

for fold_number,(fold_train_ids,fold_val_ids) in enumerate(folds,start=1):
    fold_train_df=train_df[train_df["user_id"].isin(fold_train_ids)].copy()
    fold_val_df=train_df[train_df["user_id"].isin(fold_val_ids)].copy()
    fold_datasets.append((fold_train_df,fold_val_df))
    print(f"Fold {fold_number}: "
        f"Train rows = {len(fold_train_df)}, "
        f"Validation rows = {len(fold_val_df)}")


Fold 1: Train rows = 177324, Validation rows = 42659
Fold 2: Train rows = 176273, Validation rows = 43710
Fold 3: Train rows = 170186, Validation rows = 49797
Fold 4: Train rows = 178311, Validation rows = 41672
Fold 5: Train rows = 177838, Validation rows = 42145


In [8]:
for fold_number, (fold_train_df, fold_val_df) in enumerate(
    fold_datasets, start=1
):
    train_users = set(fold_train_df["user_id"].unique())
    val_users = set(fold_val_df["user_id"].unique())

    print(
        f"Fold {fold_number}: "
        f"Overlap = {len(train_users & val_users)}, "
        f"Validation students = {len(val_users)}"
    )

Fold 1: Overlap = 0, Validation students = 675
Fold 2: Overlap = 0, Validation students = 675
Fold 3: Overlap = 0, Validation students = 675
Fold 4: Overlap = 0, Validation students = 674
Fold 5: Overlap = 0, Validation students = 674


In [9]:
print(folds)

[(array([82485, 87124, 89946, ..., 78515, 88146, 79451], shape=(2698,)), array([78492, 87142, 81458, 86717, 87459, 82931, 84396, 91977, 74459,
       90205, 79620, 80096, 86347, 90184, 91863, 80639, 83928, 85191,
       88319, 86368, 81186, 87059, 80136, 84797, 83705, 84488, 82298,
       78773, 82279, 82558, 77402, 79289, 84565, 88029, 89159, 78265,
       78532, 79632, 87038, 84374, 80237, 80906, 84321, 79726, 90505,
       84981, 85367, 80119, 91436, 90010, 89444, 82434, 85289, 83642,
       82751, 78555, 77992, 88216, 84133, 87374, 84835, 92153, 82739,
       86420, 85051, 78037, 86738, 80665, 91046, 79597, 88789, 87259,
       88020, 82770, 85722, 80255, 78887, 79565, 85696, 84612, 78704,
       92157, 79561, 78819, 83306, 80165, 78048, 91176, 89377, 88441,
       89191, 78922, 83957, 84899, 79074, 85653, 88004, 80133, 79616,
       77117, 78411, 84900, 86412, 88787, 89252, 91903, 96224, 87343,
       72735, 77961, 84408, 82802, 77957, 91985, 89162, 75169, 78708,
       85698, 919

In [10]:
from pathlib import Path

# Store each training-pool student's validation fold
fold_lookup = {}

for fold_number, (fold_train_ids, fold_val_ids) in enumerate(folds, start=1):
    for student_id in fold_val_ids:
        fold_lookup[student_id] = fold_number

# Create training-pool assignments
train_assignments = pd.DataFrame({
    "user_id": train_student_ids,
    "partition": "train_pool"
})

train_assignments["validation_fold"] = (
    train_assignments["user_id"].map(fold_lookup)
)

# Create final test assignments
test_assignments = pd.DataFrame({
    "user_id": test_student_ids,
    "partition": "test"
})

test_assignments["validation_fold"] = pd.NA

# Combine and save
split_assignments = pd.concat(
    [train_assignments, test_assignments],
    ignore_index=True
)

split_assignments["validation_fold"] = (
    split_assignments["validation_fold"].astype("Int64")
)

output_path = Path("../data/processed/split_assignments.csv")
split_assignments.to_csv(output_path, index=False)

print("Saved to:", output_path)
print(split_assignments["partition"].value_counts())
print("\nValidation fold distribution:")
print(split_assignments["validation_fold"].value_counts().sort_index())

Saved to: ..\data\processed\split_assignments.csv
partition
train_pool    3373
test           844
Name: count, dtype: int64

Validation fold distribution:
validation_fold
1    675
2    675
3    675
4    674
5    674
Name: count, dtype: Int64
